In [24]:
#data import and loading

import pandas as pd

df = pd.read_csv("Customer-Churn-Records.csv")
df.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [25]:
#understanding how the dataset is structured

import pandas as pd

df = pd.read_csv("Customer-Churn-Records.csv")
df.head()

# Basic dataset information
df.info()

# Numerical summary
df.describe()

# Check for missing values per column
print("\nMissing values per column:")
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RowNumber           10000 non-null  int64  
 1   CustomerId          10000 non-null  int64  
 2   Surname             10000 non-null  object 
 3   CreditScore         10000 non-null  int64  
 4   Geography           10000 non-null  object 
 5   Gender              10000 non-null  object 
 6   Age                 10000 non-null  int64  
 7   Tenure              10000 non-null  int64  
 8   Balance             10000 non-null  float64
 9   NumOfProducts       10000 non-null  int64  
 10  HasCrCard           10000 non-null  int64  
 11  IsActiveMember      10000 non-null  int64  
 12  EstimatedSalary     10000 non-null  float64
 13  Exited              10000 non-null  int64  
 14  Complain            10000 non-null  int64  
 15  Satisfaction Score  10000 non-null  int64  
 16  Card 

,0
RowNumber,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


In [26]:
#preparing the data: cleaning columns and defining X and y

# 1) Remove useless identifier columns
df_model = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# 2) Define target (what we want to predict)
y = df_model["Exited"]

# 3) Define features (information used to predict)
X = df_model.drop("Exited", axis=1)

# 4) Check shapes and columns
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:\n", X.columns.tolist())

print("\nFirst 5 values of y:\n", y.head())


X shape: (10000, 14)
y shape: (10000,)

X columns:
 ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Complain', 'Satisfaction Score', 'Card Type', 'Point Earned']

First 5 values of y:
 0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64


In [19]:
#train/test split (splitting data to learn and to verify the model)
from sklearn.model_selection import train_test_split

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,       # 20% test, 80% train
    random_state=42,     # for reproducibility
    stratify=y           # keep same class ratio in train and test
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (8000, 14)
X_test shape: (2000, 14)
y_train shape: (8000,)
y_test shape: (2000,)


In [27]:
#Preprocessing and model using a Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1) Define categorical and numeric columns
categorical_features = ["Geography", "Gender", "Card Type"]
numeric_features = [col for col in X.columns if col not in categorical_features]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

# 2) Preprocessor: one-hot encode categorical, pass numeric as-is
preprocessor = ColumnTransformer(
    transformers=[
        ("name", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# 3) Full pipeline: preprocessing + logistic regression model
clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,            # increased to avoid convergence warning
            class_weight="balanced"   # handle class imbalance
        ))
    ]
)

# 4) Train (fit) the model on training data
clf.fit(X_train, y_train)


Categorical features: ['Geography', 'Gender', 'Card Type']
Numeric features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Complain', 'Satisfaction Score', 'Point Earned']


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('name',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['Geography', 'Gender',
                                                   'Card Type']),
                                                 ('num', 'passthrough',
                                                  ['CreditScore', 'Age',
                                                   'Tenure', 'Balance',
                                                   'NumOfProducts', 'HasCrCard',
                                                   'IsActiveMember',
                                                   'EstimatedSalary',
                                                   'Complain',
                                                   'Satisfaction Score',
                                                   'Point Earned'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=5000))])

In [23]:
#model evaluation on the test set

from sklearn.metrics import classification_report, confusion_matrix

# Predict on test set
y_pred = clf.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))


Confusion matrix:
[[1591    1]
 [   2  406]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1592
           1       1.00      1.00      1.00       408

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

